In [1]:
import pandas as pd
from collections import Counter

# Sample dataset
data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain',
                'Overcast', 'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast',
                'Overcast', 'Rain'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool',
                    'Cool', 'Mild', 'Cool', 'Mild', 'Mild', 'Mild',
                    'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal',
                 'Normal', 'High', 'Normal', 'Normal', 'Normal', 'High',
                 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong',
             'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong',
             'Weak', 'Strong'],
    'Play': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No',
             'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes',
             'Yes', 'No']
}

df = pd.DataFrame(data)

print("Dataset:")
print(df)

Dataset:
     Outlook Temperature Humidity    Wind Play
0      Sunny         Hot     High    Weak   No
1      Sunny         Hot     High  Strong   No
2   Overcast         Hot     High    Weak  Yes
3       Rain        Mild     High    Weak  Yes
4       Rain        Cool   Normal    Weak  Yes
5       Rain        Cool   Normal  Strong   No
6   Overcast        Cool   Normal  Strong  Yes
7      Sunny        Mild     High    Weak   No
8      Sunny        Cool   Normal    Weak  Yes
9       Rain        Mild   Normal    Weak  Yes
10     Sunny        Mild   Normal  Strong  Yes
11  Overcast        Mild     High  Strong  Yes
12  Overcast         Hot   Normal    Weak  Yes
13      Rain        Mild     High  Strong   No


In [2]:
def calculate_priors(df, target):
    classes = df[target].unique()
    priors = {}

    total = len(df)

    for c in classes:
        count = sum(df[target] == c)
        priors[c] = count / total

    return priors


priors = calculate_priors(df, 'Play')

print("Prior Probabilities:")
for cls, prob in priors.items():
    print(cls, "=", prob)

Prior Probabilities:
No = 0.35714285714285715
Yes = 0.6428571428571429


In [3]:
def calculate_conditional_probabilities(df, target):
    conditional_probs = {}

    features = [col for col in df.columns if col != target]
    classes = df[target].unique()

    for feature in features:
        conditional_probs[feature] = {}

        values = df[feature].unique()

        for value in values:
            conditional_probs[feature][value] = {}

            for cls in classes:
                numerator = len(
                    df[(df[feature] == value) & (df[target] == cls)]
                )

                denominator = len(df[df[target] == cls])

                if denominator != 0:
                    probability = numerator / denominator
                else:
                    probability = 0

                conditional_probs[feature][value][cls] = probability

    return conditional_probs


conditional_probs = calculate_conditional_probabilities(df, 'Play')

print("Conditional Probabilities:\n")

for feature in conditional_probs:
    print("Feature:", feature)

    for value in conditional_probs[feature]:
        print(value, conditional_probs[feature][value])

Conditional Probabilities:

Feature: Outlook
Sunny {'No': 0.6, 'Yes': 0.2222222222222222}
Overcast {'No': 0.0, 'Yes': 0.4444444444444444}
Rain {'No': 0.4, 'Yes': 0.3333333333333333}
Feature: Temperature
Hot {'No': 0.4, 'Yes': 0.2222222222222222}
Mild {'No': 0.4, 'Yes': 0.4444444444444444}
Cool {'No': 0.2, 'Yes': 0.3333333333333333}
Feature: Humidity
High {'No': 0.8, 'Yes': 0.3333333333333333}
Normal {'No': 0.2, 'Yes': 0.6666666666666666}
Feature: Wind
Weak {'No': 0.4, 'Yes': 0.6666666666666666}
Strong {'No': 0.6, 'Yes': 0.3333333333333333}


In [4]:
def predict_naive_bayes(test_instance, priors, conditional_probs):
    probabilities = {}

    for cls in priors:
        probability = priors[cls]

        for feature, value in test_instance.items():

            # If value/class combination does not exist,
            # probability becomes zero
            if value in conditional_probs[feature]:
                cond_prob = conditional_probs[feature][value].get(cls, 0)
            else:
                cond_prob = 0

            probability *= cond_prob

        probabilities[cls] = probability

    # Select class with maximum probability
    prediction = max(probabilities, key=probabilities.get)

    return prediction, probabilities

In [5]:
test_instance = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

prediction, probabilities = predict_naive_bayes(
    test_instance,
    priors,
    conditional_probs
)

print("Test Instance:")
print(test_instance)

print("\nProbabilities:")
for cls, prob in probabilities.items():
    print(cls, "=", prob)

print("\nPredicted Class:", prediction)

Test Instance:
{'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}

Probabilities:
No = 0.02057142857142857
Yes = 0.005291005291005291

Predicted Class: No


In [6]:
def calculate_accuracy(df, target, priors, conditional_probs):
    correct = 0

    for index, row in df.iterrows():

        test_instance = {
            feature: row[feature]
            for feature in df.columns
            if feature != target
        }

        prediction, _ = predict_naive_bayes(
            test_instance,
            priors,
            conditional_probs
        )

        if prediction == row[target]:
            correct += 1

    accuracy = correct / len(df)

    return accuracy


accuracy_without = calculate_accuracy(
    df,
    'Play',
    priors,
    conditional_probs
)

print("Accuracy without Laplace smoothing:",
      accuracy_without * 100, "%")

Accuracy without Laplace smoothing: 92.85714285714286 %


In [7]:
def calculate_conditional_probabilities_laplace(df, target):
    conditional_probs = {}

    features = [col for col in df.columns if col != target]
    classes = df[target].unique()

    for feature in features:
        conditional_probs[feature] = {}

        values = df[feature].unique()
        num_values = len(values)

        for value in values:
            conditional_probs[feature][value] = {}

            for cls in classes:

                numerator = len(
                    df[(df[feature] == value) & (df[target] == cls)]
                )

                denominator = len(df[df[target] == cls])

                # Laplace smoothing
                probability = (
                    numerator + 1
                ) / (
                    denominator + num_values
                )

                conditional_probs[feature][value][cls] = probability

    return conditional_probs


conditional_probs_laplace = calculate_conditional_probabilities_laplace(
    df,
    'Play'
)

print("Conditional probabilities with Laplace smoothing:\n")

for feature in conditional_probs_laplace:
    print("Feature:", feature)

    for value in conditional_probs_laplace[feature]:
        print(value, conditional_probs_laplace[feature][value])

Conditional probabilities with Laplace smoothing:

Feature: Outlook
Sunny {'No': 0.5, 'Yes': 0.25}
Overcast {'No': 0.125, 'Yes': 0.4166666666666667}
Rain {'No': 0.375, 'Yes': 0.3333333333333333}
Feature: Temperature
Hot {'No': 0.375, 'Yes': 0.25}
Mild {'No': 0.375, 'Yes': 0.4166666666666667}
Cool {'No': 0.25, 'Yes': 0.3333333333333333}
Feature: Humidity
High {'No': 0.7142857142857143, 'Yes': 0.36363636363636365}
Normal {'No': 0.2857142857142857, 'Yes': 0.6363636363636364}
Feature: Wind
Weak {'No': 0.42857142857142855, 'Yes': 0.6363636363636364}
Strong {'No': 0.5714285714285714, 'Yes': 0.36363636363636365}


In [8]:
prediction_laplace, probabilities_laplace = predict_naive_bayes(
    test_instance,
    priors,
    conditional_probs_laplace
)

print("Test Instance:")
print(test_instance)

print("\nProbabilities with Laplace smoothing:")

for cls, prob in probabilities_laplace.items():
    print(cls, "=", prob)

print("\nPredicted Class:", prediction_laplace)

Test Instance:
{'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}

Probabilities with Laplace smoothing:
No = 0.018221574344023325
Yes = 0.007083825265643448

Predicted Class: No


In [9]:
accuracy_with = calculate_accuracy(
    df,
    'Play',
    priors,
    conditional_probs_laplace
)

print("Accuracy with Laplace smoothing:",
      accuracy_with * 100, "%")

Accuracy with Laplace smoothing: 92.85714285714286 %


In [10]:
print("======================================")
print("Naïve Bayes Performance Comparison")
print("======================================")

print("Accuracy without Laplace smoothing:",
      round(accuracy_without * 100, 2), "%")

print("Accuracy with Laplace smoothing:",
      round(accuracy_with * 100, 2), "%")

if accuracy_with > accuracy_without:
    print("\nLaplace smoothing improved the accuracy.")

elif accuracy_with < accuracy_without:
    print("\nLaplace smoothing reduced the accuracy on this dataset.")

else:
    print("\nBoth methods produced the same accuracy.")

Naïve Bayes Performance Comparison
Accuracy without Laplace smoothing: 92.86 %
Accuracy with Laplace smoothing: 92.86 %

Both methods produced the same accuracy.
